In [4]:
# ============================================================
# RASTERIZE BIOKLIMA NUMERIC POLYGON ATTRIBUTES TO ALIGNED COGs
#
# Input polygon:
#   Bioklima.shp
#
# Numeric fields:
#   PCA1
#   PCA2
#
# Outputs:
#   BioK_PCA1.tif
#   BioK_PCA2.tif
#
# Requirements:
#   - Same CRS as mask
#   - Same extent
#   - Same cell size
#   - Same alignment
#   - Masked by mask raster
#   - Saved as Cloud Optimized GeoTIFFs
# ============================================================

import arcpy
import os
import time
from arcpy.sa import ExtractByMask

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

BIOKLIMA_SHP = (
    r"C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse"
    r"\Geochemistry NGU_2025\kalk_prosjekt3.0\MDir_data\Data\Data"
    r"\BioK_shape\Bioklima.shp"
)

MASK_RASTER = r"E:\mask\Mask_land_Kalk_cog.tif"

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------

OUT_DIR = r"G:\Geology_rasters_2026\BioKlima"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PCA1_COG = os.path.join(OUT_DIR, "BioK_PCA1.tif")
OUT_PCA2_COG = os.path.join(OUT_DIR, "BioK_PCA2.tif")

# ------------------------------------------------------------
# FIELDS
# ------------------------------------------------------------

PCA1_FIELD = "PCA1"
PCA2_FIELD = "PCA2"

FIELDS_TO_RASTERIZE = [
    (PCA1_FIELD, OUT_PCA1_COG),
    (PCA2_FIELD, OUT_PCA2_COG)
]

# ------------------------------------------------------------
# LOCAL TEMP WORKSPACE
# ------------------------------------------------------------

LOCAL_TMP_DIR = r"C:\temp\rasterize_bioklima"
os.makedirs(LOCAL_TMP_DIR, exist_ok=True)

TMP_GDB = os.path.join(LOCAL_TMP_DIR, "temp_bioklima.gdb")

if arcpy.Exists(TMP_GDB):
    arcpy.management.Delete(TMP_GDB)

arcpy.management.CreateFileGDB(LOCAL_TMP_DIR, "temp_bioklima.gdb")

# ------------------------------------------------------------
# ENVIRONMENT FROM MASK
# ------------------------------------------------------------

mask_desc = arcpy.Describe(MASK_RASTER)

arcpy.env.workspace = TMP_GDB
arcpy.env.scratchWorkspace = TMP_GDB
arcpy.env.snapRaster = MASK_RASTER
arcpy.env.cellSize = MASK_RASTER
arcpy.env.extent = MASK_RASTER
arcpy.env.outputCoordinateSystem = mask_desc.spatialReference
arcpy.env.mask = MASK_RASTER

cell_size = float(
    arcpy.management.GetRasterProperties(
        MASK_RASTER,
        "CELLSIZEX"
    ).getOutput(0)
)

print("Mask raster:", MASK_RASTER)
print("CRS:", mask_desc.spatialReference.name)
print("Cell size:", cell_size)
print("Rows:", arcpy.management.GetRasterProperties(MASK_RASTER, "ROWCOUNT").getOutput(0))
print("Cols:", arcpy.management.GetRasterProperties(MASK_RASTER, "COLUMNCOUNT").getOutput(0))

# ------------------------------------------------------------
# CHECK INPUT
# ------------------------------------------------------------

if not arcpy.Exists(BIOKLIMA_SHP):
    raise FileNotFoundError(f"Input shapefile not found:\n{BIOKLIMA_SHP}")

fields = arcpy.ListFields(BIOKLIMA_SHP)
field_names = [f.name for f in fields]

print("\nAvailable fields:")
for f in field_names:
    print(" -", f)

for field_name, _ in FIELDS_TO_RASTERIZE:

    if field_name not in field_names:
        raise ValueError(f"Field '{field_name}' not found in:\n{BIOKLIMA_SHP}")

    field_obj = [f for f in fields if f.name == field_name][0]

    if field_obj.type not in ["Double", "Single", "Integer", "SmallInteger"]:
        raise TypeError(
            f"Field '{field_name}' is not numeric. "
            f"Current type: {field_obj.type}"
        )

# ------------------------------------------------------------
# PROJECT TO MASK CRS IF NEEDED
# ------------------------------------------------------------

input_desc = arcpy.Describe(BIOKLIMA_SHP)
projected_fc = os.path.join(TMP_GDB, "bioklima_projected")

if input_desc.spatialReference.name != mask_desc.spatialReference.name:
    print("\nProjecting Bioklima polygons to mask CRS...")

    arcpy.management.Project(
        in_dataset=BIOKLIMA_SHP,
        out_dataset=projected_fc,
        out_coor_system=mask_desc.spatialReference
    )

    work_fc = projected_fc

else:
    print("\nInput already has same CRS as mask.")
    work_fc = BIOKLIMA_SHP

# ------------------------------------------------------------
# COPY LOCALLY
# ------------------------------------------------------------

local_fc = os.path.join(TMP_GDB, "bioklima_local")

print("\nCopying polygons to local temporary geodatabase...")
arcpy.management.CopyFeatures(work_fc, local_fc)

# ------------------------------------------------------------
# FUNCTION TO RASTERIZE ONE NUMERIC FIELD
# ------------------------------------------------------------

def rasterize_numeric_field(in_fc, value_field, out_cog):

    tmp_raster = os.path.join(TMP_GDB, f"{value_field}_raster")

    if arcpy.Exists(tmp_raster):
        arcpy.management.Delete(tmp_raster)

    print("\n" + "=" * 70)
    print(f"Rasterizing field: {value_field}")
    print("=" * 70)

    start = time.time()

    arcpy.conversion.PolygonToRaster(
        in_features=in_fc,
        value_field=value_field,
        out_rasterdataset=tmp_raster,
        cell_assignment="MAXIMUM_AREA",
        priority_field="NONE",
        cellsize=cell_size
    )

    print(f"Rasterization completed in {(time.time() - start) / 60:.1f} min")

    print("Masking raster...")
    start = time.time()

    masked_raster = ExtractByMask(tmp_raster, MASK_RASTER)

    print(f"Masking completed in {(time.time() - start) / 60:.1f} min")

    if arcpy.Exists(out_cog):
        arcpy.management.Delete(out_cog)

    print("Saving COG:")
    print(out_cog)

    start = time.time()

    arcpy.management.CopyRaster(
        in_raster=masked_raster,
        out_rasterdataset=out_cog,
        pixel_type="32_BIT_FLOAT",
        nodata_value="-9999",
        format="COG"
    )

    print(f"COG saved in {(time.time() - start) / 60:.1f} min")
    print("Saved:", out_cog)

# ------------------------------------------------------------
# RUN PCA1 AND PCA2
# ------------------------------------------------------------

overall_start = time.time()

for field_name, out_cog in FIELDS_TO_RASTERIZE:
    rasterize_numeric_field(
        in_fc=local_fc,
        value_field=field_name,
        out_cog=out_cog
    )

print("\nDone.")
print(f"Total time: {(time.time() - overall_start) / 60:.1f} min")
print("Outputs:")
print(" -", OUT_PCA1_COG)
print(" -", OUT_PCA2_COG)

Mask raster: E:\mask\Mask_land_Kalk_cog.tif
CRS: ETRS_1989_UTM_Zone_33N
Cell size: 10.0
Rows: 162232
Cols: 135236

Available fields:
 - FID
 - Shape
 - OBJECTID_1
 - SSBID
 - PCA1
 - Seksjon_ko
 - Seksjon_na
 - Dataeier
 - COUNT
 - SHAPE_Leng
 - SHAPE_Area
 - SSBID_1
 - PCA2
 - Sone_kode
 - Sone_navn
 - Dataeier_1
 - Shape_Le_1
 - Shape_Ar_1
 - SoneSeksjo

Projecting Bioklima polygons to mask CRS...

Copying polygons to local temporary geodatabase...

Rasterizing field: PCA1
Rasterization completed in 4.8 min
Masking raster...
Masking completed in 16.0 min
Saving COG:
G:\Geology_rasters_2026\BioKlima\BioK_PCA1.tif
COG saved in 12.0 min
Saved: G:\Geology_rasters_2026\BioKlima\BioK_PCA1.tif

Rasterizing field: PCA2
Rasterization completed in 6.7 min
Masking raster...
Masking completed in 17.2 min
Saving COG:
G:\Geology_rasters_2026\BioKlima\BioK_PCA2.tif
COG saved in 12.0 min
Saved: G:\Geology_rasters_2026\BioKlima\BioK_PCA2.tif

Done.
Total time: 68.7 min
Outputs:
 - G:\Geology_rasters_